In [ ]:
# Strategy pattern using class


from typing import NamedTuple
from dataclasses import dataclass
from abc import ABC, abstractmethod
from collections.abc import Sequence

class Customer(NamedTuple):
    name: str
    fidelity: int


class Item(NamedTuple):
    name: str
    qty: int
    price: float

    def total(self) -> float:
        return self.price * self.qty


class Order(NamedTuple):
    customer: Customer
    items: Sequence[Item]
    promition: Promotion | None = None

    def total(self) -> float:
        return sum(item.total() for item in self.items)

    def due(self) -> float:
        discount = self.promition.discount(self) if self.promition is not None else 0
        return self.total() - discount

    def __repr__(self):
        return f'<Order total:{self.total():.2f}, due:{self.due():.2f}>'



class Promotion(ABC):
    @abstractmethod
    def discount(self, order: Order) -> float:
        """calculate a discount for the promotion"""

class FidelityPromotion(Promotion):
    '''Get a discount with Fidelity point, default of 1000'''
    def __init__(self, rate = .05, point=1000):
        self.rate = rate
        self.point = point

    def discount(self, order: Order) -> float:
        if order.customer.fidelity >= self.point:
            return order.total()*self.rate
        return 0

class BulkPromotion(Promotion):
    '''Get a {rate}, default = 10%, discount with bulk purchase; default to 20 or more'''
    def __init__(self, count=20, rate = .1):
        self.count = count
        self.rate = rate

    def discount(self, order: Order) -> float:
        res = 0
        for item in order.items:
            if item.qty >= self.count:
                res += item.total()*self.rate
        return res

class LargeOrderPromotion(Promotion):
    '''Get a {rate}, default=7%, discount with {count}, default to 10, or more distinct items in an order'''
    def __init__(self, rate=.07, count=10):
        self.rate = rate
        self.count = count

    def discount(self, order: Order) -> float:
        #NOTE: whether distinct items is based on item name itself or (name, qty, price) is a business logic!
        #if len({item for item in order.items}) >= self.count:
        if len({item.name for item in order.items}) >= self.count:
            return order.total()*self.rate
        return 0.0

joe = Customer('joe', 10)
anna = Customer('anna', 1100)
cart = (Item('apple', 10, 1.5), Item('banana', 4, .5), Item('watermelon', 5, 5.0))
o1 = Order(joe, cart, FidelityPromotion())        
print(o1)
o2 = Order(anna, cart, FidelityPromotion())
print(o2)

print()

banana_cart = (Item('banana', 30, .5), Item('apple', 10, 1.5))
o3 = Order(joe, banana_cart, BulkPromotion())
print(o3)

long_cart = tuple(Item(str(sku), 1, 1.0) for sku in range(10))
o4 = Order(joe, long_cart, LargeOrderPromotion())
print(o4)

<Order total:42.00, due:42.00>
<Order total:42.00, due:39.90>

<Order total:30.00, due:28.50>
<Order total:10.00, due:9.30>


In [27]:
from typing import Callable, Self

@dataclass(frozen=True)
class FunctionOrder:
    customer: Customer
    items: Sequence[Item]
    promotion: Callable[[Self], float] | None = None
    def total(self) -> float:
        return sum(item.total() for item in self.items)

    def due(self) -> float:
        discount = 0 if self.promotion is None else self.promotion(self)
        return self.total() - discount

    def __repr__(self):
        return f'<{self.customer.name} - Order total:{self.total():.2f}, due:{self.due():.2f}>'

def fidelity_promotion(order: FunctionOrder, point=1000, discount_rate = .05) -> float:
    if order.customer.fidelity >= point:
        return order.total()*discount_rate
    return 0

def bulk_promotion(order: FunctionOrder, count=20, discount_rate = 0.1) -> float:
    '''Bulk order promotion for each item with 20 or more gets a discount of 10%'''
    return sum(item.total()*discount_rate for item in order.items if item.qty >= count)
    
def large_promotion(order: FunctionOrder, count=10, discount_rate=.07) -> float:
    '''order with 10 or more items gets 7% discount'''
    if len({item.name for item in order.items}) >= count:
            return order.total()*discount_rate
    return 0
    
o5 = FunctionOrder(joe, cart, fidelity_promotion)
print(o5)
o6 = FunctionOrder(anna, cart, fidelity_promotion)
print(o6)
o7 = FunctionOrder(joe, banana_cart, bulk_promotion)
print(o7)

long_cart = tuple(Item(str(sku), 1, 1.0) for sku in range(10))
o8 = FunctionOrder(joe, long_cart, large_promotion)
print(o8)

o9 = FunctionOrder(joe, cart, large_promotion)
print(o9)

<joe - Order total:42.00, due:42.00>
<anna - Order total:42.00, due:39.90>
<joe - Order total:30.00, due:28.50>
<joe - Order total:10.00, due:9.30>
<joe - Order total:42.00, due:42.00>


In [ ]:
Promotion = Callable[[FunctionOrder], float] # This is like defining a delegate in C#
promos: list[Promotion] = []

def promotion(promo: Promotion) -> Promotion:
    promos.append(promo)
    return promo

def best_promo(order: FunctionOrder) -> float:
    '''Find the best promotion for a given order'''
    return max(p(order) for p in promos)

@promotion
def fidelity(order: FunctionOrder, point=1000, rate=0.05) -> float:
    if order.customer.fidelity >= point:
        return order.total()*rate
    return 0
